# Chapter 6: Sequence Modeling & Recurrence

This chapter explores the mathematical foundations of sequence models — from classical RNNs through LSTM and GRU, to modern state-space models (S4, Mamba) and cross-attention in encoder-decoder architectures.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")


## 6.1 Vanilla RNN

A Recurrent Neural Network maintains a hidden state $\mathbf{h}_t$ that is updated at each time step:

$$\mathbf{h}_t = \tanh\!\left(\mathbf{W}_h\,\mathbf{h}_{t-1} + \mathbf{W}_x\,\mathbf{x}_t + \mathbf{b}\right)$$

**Backpropagation Through Time (BPTT)** unrolls the network and computes gradients as:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

Each factor involves $\mathbf{W}_h^\top \cdot \text{diag}(\tanh')$. When the product of $T$ such matrices shrinks to zero (vanishing) or grows without bound (exploding), learning long-range dependencies becomes impossible.


In [ ]:
torch.manual_seed(42)

# Manual RNN cell
class ManualRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Wx = nn.Linear(input_size,  hidden_size, bias=True)

    def forward(self, x, h):
        return torch.tanh(self.Wh(h) + self.Wx(x))

INPUT_SIZE, HIDDEN_SIZE, SEQ_LEN, BATCH = 8, 16, 10, 4
x_seq = torch.randn(BATCH, SEQ_LEN, INPUT_SIZE)    # (B, T, input)

# Manual RNN forward
manual_cell = ManualRNNCell(INPUT_SIZE, HIDDEN_SIZE)
h_manual = torch.zeros(BATCH, HIDDEN_SIZE)
manual_outputs = []
for t in range(SEQ_LEN):
    h_manual = manual_cell(x_seq[:, t, :], h_manual)
    manual_outputs.append(h_manual)
manual_out = torch.stack(manual_outputs, dim=1)   # (B, T, H)

# nn.RNN forward (copy weights)
rnn = nn.RNN(INPUT_SIZE, HIDDEN_SIZE, batch_first=True)
with torch.no_grad():
    rnn.weight_hh_l0.copy_(manual_cell.Wh.weight)
    rnn.weight_ih_l0.copy_(manual_cell.Wx.weight)
    rnn.bias_ih_l0.copy_(manual_cell.Wx.bias)
    rnn.bias_hh_l0.zero_()

nn_out, _ = rnn(x_seq)

max_diff = (manual_out - nn_out).abs().max().item()
print(f"Manual RNN output shape : {manual_out.shape}")
print(f"nn.RNN  output shape    : {nn_out.shape}")
print(f"Max absolute difference : {max_diff:.2e}  (should be ~0)")

# Gradient norm over 50 steps to show vanishing
rnn50 = nn.RNN(INPUT_SIZE, HIDDEN_SIZE, batch_first=True)
x50   = torch.randn(1, 50, INPUT_SIZE, requires_grad=True)
out50, _ = rnn50(x50)
loss50 = out50[:, -1, :].sum()   # gradient w.r.t. each time step
loss50.backward()
print("\nGradient norm w.r.t. input at selected time steps (50-step RNN):")
for t in [0, 10, 25, 40, 49]:
    gnorm = x50.grad[:, t, :].norm().item()
    print(f"  t={t:>3}: grad norm = {gnorm:.4e}")


## 6.2 Vanishing Gradient Demo

The gradient of the loss at step $T$ with respect to the hidden state at step $t$ is:

$$\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_t} = \prod_{k=t+1}^{T} \mathbf{W}_h^\top \cdot \text{diag}(\sigma'(\mathbf{a}_k))$$

This product vanishes at exponential rate if $\rho(\mathbf{W}_h) \cdot \max|\sigma'| < 1$:

$$\left\|\frac{\partial \mathbf{h}_T}{\partial \mathbf{h}_t}\right\| \approx \left(\rho(\mathbf{W}_h) \cdot \max|\sigma'|\right)^{T - t - 1}$$

For $\tanh$, $\max|\sigma'| = 1$, so the spectral radius of $\mathbf{W}_h$ determines stability. LSTMs mitigate this via the cell state, which maintains an approximately linear gradient highway.


In [ ]:
torch.manual_seed(42)

INPUT_SIZE, HIDDEN, SEQ = 4, 32, 100
x_long = torch.randn(1, SEQ, INPUT_SIZE, requires_grad=True)

rnn_model  = nn.RNN(INPUT_SIZE,  HIDDEN, batch_first=True)
lstm_model = nn.LSTM(INPUT_SIZE, HIDDEN, batch_first=True)

# RNN gradient norms
out_rnn, _ = rnn_model(x_long)
out_rnn[:, -1, :].sum().backward()
rnn_grad_norms = [x_long.grad[:, t, :].norm().item() for t in range(SEQ)]

# LSTM gradient norms (fresh graph)
x_long2 = torch.randn(1, SEQ, INPUT_SIZE, requires_grad=True)
out_lstm, _ = lstm_model(x_long2)
out_lstm[:, -1, :].sum().backward()
lstm_grad_norms = [x_long2.grad[:, t, :].norm().item() for t in range(SEQ)]

print(f"{'Step':>6} {'RNN grad norm':>16} {'LSTM grad norm':>16}")
print("-" * 42)
for t in [0, 10, 25, 50, 75, 99]:
    print(f"{t:>6} {rnn_grad_norms[t]:>16.4e} {lstm_grad_norms[t]:>16.4e}")

# Spectral radius of RNN weight matrix
W_hh = rnn_model.weight_hh_l0.detach()
eigenvalues = torch.linalg.eigvals(W_hh)
spectral_radius = eigenvalues.abs().max().item()
print(f"\nRNN W_h spectral radius: {spectral_radius:.4f}")
print(f"Vanishing expected if spectral radius * max|tanh'| = {spectral_radius:.4f} < 1: {spectral_radius < 1}")


## 6.3 Long Short-Term Memory (LSTM)

LSTMs (Hochreiter & Schmidhuber, 1997) introduce a **cell state** $\mathbf{c}_t$ that flows through time with only element-wise multiplications — providing a near-constant gradient highway.

The four gate equations (using $[\mathbf{h}_{t-1}, \mathbf{x}_t]$ to denote concatenation):

$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(candidate cell)}$$
$$\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$

The cell state update and hidden state:

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

The gradient of $\mathbf{c}_t$ with respect to $\mathbf{c}_{t-1}$ is simply $\mathbf{f}_t$ — a multiplicative factor that, when close to 1, preserves gradients over long distances.


In [ ]:
torch.manual_seed(42)

class ManualLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        # Combined weight matrix for all four gates
        self.W_ih = nn.Linear(input_size,  4 * hidden_size, bias=True)
        self.W_hh = nn.Linear(hidden_size, 4 * hidden_size, bias=False)
        self.H = hidden_size

    def forward(self, x, h, c):
        gates = self.W_ih(x) + self.W_hh(h)   # (B, 4H)
        i, f, g, o = gates.chunk(4, dim=-1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        g = torch.tanh(g)
        o = torch.sigmoid(o)
        c_new = f * c + i * g
        h_new = o * torch.tanh(c_new)
        return h_new, c_new

INPUT_S, HIDDEN_S, SEQ_L, BATCH_S = 8, 16, 6, 3
x_seq = torch.randn(BATCH_S, SEQ_L, INPUT_S)

lstm_nn = nn.LSTM(INPUT_S, HIDDEN_S, batch_first=True)

# Copy weights into manual cell
cell = ManualLSTMCell(INPUT_S, HIDDEN_S)
with torch.no_grad():
    cell.W_ih.weight.copy_(lstm_nn.weight_ih_l0)
    cell.W_ih.bias.copy_(lstm_nn.bias_ih_l0 + lstm_nn.bias_hh_l0)
    cell.W_hh.weight.copy_(lstm_nn.weight_hh_l0)

# nn.LSTM forward
nn_out, (hn_nn, cn_nn) = lstm_nn(x_seq)

# Manual LSTM forward
h, c = torch.zeros(BATCH_S, HIDDEN_S), torch.zeros(BATCH_S, HIDDEN_S)
manual_outputs = []
for t in range(SEQ_L):
    h, c = cell(x_seq[:, t, :], h, c)
    manual_outputs.append(h)
manual_out = torch.stack(manual_outputs, dim=1)

diff = (manual_out - nn_out).abs().max().item()
print(f"Manual LSTM output shape : {manual_out.shape}")
print(f"nn.LSTM output shape     : {nn_out.shape}")
print(f"Max absolute difference  : {diff:.2e}  (should be ~0)")


## 6.4 Gated Recurrent Unit (GRU)

GRUs (Cho et al., 2014) simplify the LSTM by merging the forget and input gates into a single **update gate** $\mathbf{z}_t$:

$$\mathbf{z}_t = \sigma(\mathbf{W}_z [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_z)  \quad \text{(update gate)}$$
$$\mathbf{r}_t = \sigma(\mathbf{W}_r [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_r)  \quad \text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(\mathbf{W}_h [\mathbf{r}_t \odot \mathbf{h}_{t-1},\, \mathbf{x}_t] + \mathbf{b}_h) \quad \text{(candidate)}$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

When $\mathbf{z}_t \approx 0$, the previous hidden state is copied unchanged. When $\mathbf{z}_t \approx 1$, the hidden state is fully replaced. GRUs use $3H^2$ weight parameters per layer vs LSTMs' $4H^2$, making them faster to train at similar performance.


In [ ]:
torch.manual_seed(42)

INPUT_DIM, HIDDEN_DIM = 32, 128

rnn_model  = nn.RNN(INPUT_DIM,  HIDDEN_DIM, batch_first=True)
lstm_model = nn.LSTM(INPUT_DIM, HIDDEN_DIM, batch_first=True)
gru_model  = nn.GRU(INPUT_DIM,  HIDDEN_DIM, batch_first=True)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

rnn_params  = count_params(rnn_model)
lstm_params = count_params(lstm_model)
gru_params  = count_params(gru_model)

print(f"Architecture parameter comparison (input={INPUT_DIM}, hidden={HIDDEN_DIM}):")
print(f"  RNN  params : {rnn_params:>8}")
print(f"  LSTM params : {lstm_params:>8}")
print(f"  GRU  params : {gru_params:>8}")
print(f"\n  LSTM / RNN  : {lstm_params / rnn_params:.2f}x")
print(f"  GRU  / RNN  : {gru_params  / rnn_params:.2f}x")
print(f"  LSTM / GRU  : {lstm_params / gru_params:.2f}x")

# Theoretical gate counts
H, I = HIDDEN_DIM, INPUT_DIM
print(f"\nTheoretical weight counts:")
print(f"  RNN  : 1 gate  = (H+I)*H + H  = {(H+I)*H + H}")
print(f"  LSTM : 4 gates = 4*(H+I)*H + 4H = {4*(H+I)*H + 4*H}")
print(f"  GRU  : 3 gates = 3*(H+I)*H + 3H = {3*(H+I)*H + 3*H}")


## 6.5 S4 Linear Recurrence (State Space Models)

**Structured State Space Sequence (S4)** models (Gu et al., 2021) are grounded in continuous-time linear dynamical systems:

$$\dot{\mathbf{h}}(t) = \mathbf{A}\,\mathbf{h}(t) + \mathbf{B}\,x(t), \quad y(t) = \mathbf{C}\,\mathbf{h}(t)$$

**Discretisation via Zero-Order Hold (ZOH)** with step size $\Delta$:

$$\bar{\mathbf{A}} = e^{\Delta \mathbf{A}}, \quad \bar{\mathbf{B}} = (\mathbf{A})^{-1}(e^{\Delta\mathbf{A}} - \mathbf{I})\mathbf{B}$$

The resulting discrete recurrence is:
$$\mathbf{h}_k = \bar{\mathbf{A}}\,\mathbf{h}_{k-1} + \bar{\mathbf{B}}\,x_k, \quad y_k = \mathbf{C}\,\mathbf{h}_k$$

**Key insight**: the same computation can be expressed as a **convolution** with kernel $\mathbf{K} = (\mathbf{C}\bar{\mathbf{B}},\, \mathbf{C}\bar{\mathbf{A}}\bar{\mathbf{B}},\, \mathbf{C}\bar{\mathbf{A}}^2\bar{\mathbf{B}}, \ldots)$. This allows $O(T\log T)$ training via FFT while maintaining $O(1)$ per-step inference as a recurrence.


In [ ]:
torch.manual_seed(42)

class SimpleS4Layer(nn.Module):
    """Simplified S4-style layer with diagonal A for clarity."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Diagonal A initialised to have negative real parts (stable)
        log_A_real = torch.log(0.5 * torch.ones(d_model, d_state))
        self.log_A_real = nn.Parameter(log_A_real)

        self.B = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.C = nn.Parameter(torch.randn(d_model, d_state) * 0.1)
        self.log_dt = nn.Parameter(torch.zeros(d_model))   # log step size

    def _get_discrete(self):
        dt = torch.exp(self.log_dt).unsqueeze(-1)          # (D, 1)
        A  = -torch.exp(self.log_A_real)                   # (D, N) diagonal, negative
        A_bar = torch.exp(dt * A)                          # ZOH: exp(dt * A)
        B_bar = (1 - A_bar) / (A + 1e-8) * self.B         # (A^{-1})(exp(dtA)-I)B
        return A_bar, B_bar

    def forward_recurrence(self, x):
        """Recurrence mode: O(T). x: (B, T, D)"""
        B_sz, T, D = x.shape
        A_bar, B_bar = self._get_discrete()               # (D, N), (D, N)
        h = torch.zeros(B_sz, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            xt = x[:, t, :]                               # (B, D)
            h = A_bar.unsqueeze(0) * h + B_bar.unsqueeze(0) * xt.unsqueeze(-1)
            y = (h * self.C.unsqueeze(0)).sum(-1)         # (B, D)
            ys.append(y)
        return torch.stack(ys, dim=1)                     # (B, T, D)

    def forward_convolution(self, x):
        """Convolution mode: O(T log T) via FFT. x: (B, T, D)"""
        B_sz, T, D = x.shape
        A_bar, B_bar = self._get_discrete()

        # Build convolution kernel K of length T
        # K_t = C * A_bar^t * B_bar  (scalar per channel)
        K = torch.zeros(T, D, device=x.device)
        A_pow = torch.ones_like(A_bar)                    # A_bar^0
        CB = (self.C * B_bar)                             # (D, N)
        for t in range(T):
            K[t] = (CB * A_pow).sum(-1)                   # (D,)
            A_pow = A_pow * A_bar
        K = K.T                                           # (D, T)

        # Causal convolution via FFT (zero-pad to 2T)
        x_t = x.permute(0, 2, 1)                         # (B, D, T)
        fft_size = 2 * T
        X_f = torch.fft.rfft(x_t, n=fft_size)
        K_f = torch.fft.rfft(K,   n=fft_size)
        Y_f = X_f * K_f.unsqueeze(0)
        y   = torch.fft.irfft(Y_f, n=fft_size)[..., :T]  # (B, D, T)
        return y.permute(0, 2, 1)                         # (B, T, D)

    def forward(self, x):
        return self.forward_recurrence(x)

# Verify both modes produce same output
layer = SimpleS4Layer(d_model=8, d_state=4)
x_test = torch.randn(2, 12, 8)

y_rec  = layer.forward_recurrence(x_test)
y_conv = layer.forward_convolution(x_test)

diff = (y_rec - y_conv).abs().max().item()
print(f"Recurrence output shape     : {y_rec.shape}")
print(f"Convolution output shape    : {y_conv.shape}")
print(f"Max difference (rec vs conv): {diff:.2e}  (should be ~0)")


## 6.6 Mamba Selective State Space Model

Mamba (Gu & Dao, 2023) extends S4 with **input-dependent** (selective) parameters. Instead of fixed $\mathbf{B}$, $\mathbf{C}$, and $\Delta$, these are computed from the input token:

$$\Delta_t = \text{softplus}(\mathbf{W}_\Delta\, \mathbf{x}_t)$$
$$\mathbf{B}_t = \mathbf{W}_B\, \mathbf{x}_t, \quad \mathbf{C}_t = \mathbf{W}_C\, \mathbf{x}_t$$

The state transition then becomes time-varying:
$$\mathbf{h}_t = \bar{\mathbf{A}}_t\, \mathbf{h}_{t-1} + \bar{\mathbf{B}}_t\, x_t$$

This **selection mechanism** allows the model to selectively remember or forget content based on the input, analogous to a soft attention but at $O(T)$ cost vs the Transformer's $O(T^2)$ attention.


In [ ]:
torch.manual_seed(42)

class SelectiveSSMCell(nn.Module):
    """Simplified Mamba-style selective SSM layer."""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        # Input-dependent projections
        self.W_dt = nn.Linear(d_model, d_model, bias=True)    # for Delta
        self.W_B  = nn.Linear(d_model, d_state, bias=False)   # for B
        self.W_C  = nn.Linear(d_model, d_state, bias=False)   # for C

        # Fixed A (diagonal, log-parameterised for positivity)
        self.log_A = nn.Parameter(-0.5 * torch.ones(d_model, d_state))

        # Output projection
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        """
        x: (B, T, d_model)
        returns: (B, T, d_model)
        """
        B, T, D = x.shape
        N = self.d_state
        A = -torch.exp(self.log_A)   # (D, N) negative diagonal

        h = torch.zeros(B, D, N, device=x.device)
        outputs = []

        for t in range(T):
            xt = x[:, t, :]              # (B, D)

            # Input-dependent parameters (the "selection" mechanism)
            dt = F.softplus(self.W_dt(xt))         # (B, D)  — step size
            Bt = self.W_B(xt)                       # (B, N)
            Ct = self.W_C(xt)                       # (B, N)

            # Discretise A with input-dependent dt
            # A_bar shape: (B, D, N)
            A_bar = torch.exp(dt.unsqueeze(-1) * A.unsqueeze(0))  # (B, D, N)
            B_bar = (1 - A_bar) / (A.unsqueeze(0).abs() + 1e-8) * Bt.unsqueeze(1)  # (B, D, N)

            # State update
            h = A_bar * h + B_bar * xt.unsqueeze(-1)   # (B, D, N)

            # Output: y_t = C_t^T h_t  (summed over state dim)
            y = (h * Ct.unsqueeze(1)).sum(-1)           # (B, D)
            outputs.append(y)

        out = torch.stack(outputs, dim=1)   # (B, T, D)
        return self.out_proj(out)

# Verify shapes
mamba_layer = SelectiveSSMCell(d_model=16, d_state=8)
x_in = torch.randn(2, 20, 16)   # (B=2, T=20, D=16)
y_out = mamba_layer(x_in)

print(f"Mamba SSM input  shape : {x_in.shape}   (B, T, d_model)")
print(f"Mamba SSM output shape : {y_out.shape}  (B, T, d_model)")
print(f"Input-dependent Delta  : computed per token via W_dt projection")
print(f"Complexity             : O(T) recurrence vs O(T^2) attention")


## 6.7 Encoder-Decoder Cross-Attention

In encoder-decoder architectures (e.g., the original Transformer for machine translation), the decoder attends to the encoder's output via **cross-attention**:

$$\text{CrossAttn}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}\right) \mathbf{V}$$

where:
- $\mathbf{Q} = \mathbf{H}_{\text{dec}}\,\mathbf{W}_Q$ — queries come from the **decoder** hidden states
- $\mathbf{K} = \mathbf{H}_{\text{enc}}\,\mathbf{W}_K$ — keys come from the **encoder** output
- $\mathbf{V} = \mathbf{H}_{\text{enc}}\,\mathbf{W}_V$ — values come from the **encoder** output

This allows each decoder position to attend over all encoder positions, enabling the model to align source and target tokens (e.g., words in translation).


In [ ]:
torch.manual_seed(42)

class CrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)

    def forward(self, dec_hidden, enc_output):
        """
        dec_hidden : (B, T_dec, d_model)  — decoder hidden states
        enc_output : (B, S_enc, d_model)  — encoder output
        returns    : (B, T_dec, d_model)
        """
        B, T, D = dec_hidden.shape
        _, S, _ = enc_output.shape
        H, dk = self.num_heads, self.d_k

        # Project queries from decoder, keys/values from encoder
        Q = self.W_q(dec_hidden).view(B, T, H, dk).transpose(1, 2)  # (B, H, T, dk)
        K = self.W_k(enc_output).view(B, S, H, dk).transpose(1, 2)  # (B, H, S, dk)
        V = self.W_v(enc_output).view(B, S, H, dk).transpose(1, 2)  # (B, H, S, dk)

        # Scaled dot-product attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(dk)          # (B, H, T, S)
        attn   = F.softmax(scores, dim=-1)                           # (B, H, T, S)
        ctx    = (attn @ V).transpose(1, 2).contiguous()             # (B, T, H, dk)
        ctx    = ctx.view(B, T, D)                                   # (B, T, D)
        return self.W_o(ctx)

# Demo
B, S_enc, T_dec, D_MODEL = 2, 10, 6, 64
enc_out  = torch.randn(B, S_enc, D_MODEL)    # encoder produces S=10 hidden states
dec_in   = torch.randn(B, T_dec, D_MODEL)    # decoder at T=6 steps

cross_attn = CrossAttention(d_model=D_MODEL, num_heads=8)
output = cross_attn(dec_in, enc_out)

print(f"Encoder output shape   : {enc_out.shape}  (B, S_enc, d)")
print(f"Decoder input shape    : {dec_in.shape}   (B, T_dec, d)")
print(f"Cross-attn output shape: {output.shape}   (B, T_dec, d)")
print(f"Q from decoder | K,V from encoder — confirmed by shapes above")


## 6.8 Architecture Comparison

| Architecture | Time Complexity | Memory | Training | Inference | Long-range deps |
|---|---|---|---|---|---|
| Transformer | $O(T^2 \cdot d)$ | $O(T^2 + T \cdot d)$ | Parallel | Parallel (KV cache) | Excellent |
| LSTM / GRU | $O(T \cdot d^2)$ | $O(T \cdot d)$ | Sequential | Sequential | Good (gating) |
| S4 | $O(T \log T \cdot d)$ | $O(T \cdot d)$ | Parallel (FFT) | Recurrence $O(1)$/step | Excellent |
| Mamba (SSM) | $O(T \cdot d \cdot N)$ | $O(T \cdot d)$ | Parallel scan | Recurrence $O(1)$/step | Excellent |

Key trade-offs:
- **Transformers** dominate quality on large-scale pretraining but suffer quadratic attention cost for long sequences.
- **LSTMs/GRUs** are memory-efficient and fast at inference but cannot parallelise training.
- **Mamba** and S4 achieve Transformer-quality long-range modelling at linear cost, making them attractive for long-context tasks.


In [ ]:
import time
torch.manual_seed(42)

SEQ_LEN, D_MODEL, BATCH_SIZE, NUM_HEADS = 512, 64, 4, 8
x_bench = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)

rnn_bench   = nn.RNN(D_MODEL, D_MODEL, batch_first=True)
lstm_bench  = nn.LSTM(D_MODEL, D_MODEL, batch_first=True)

# Simple self-attention layer for comparison
class SimpleAttentionLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)

    def forward(self, x):
        out, _ = self.attn(x, x, x)
        return out

attn_bench = SimpleAttentionLayer(D_MODEL, NUM_HEADS)

N_RUNS = 10

def bench(model, x, n=N_RUNS):
    # Warmup
    with torch.no_grad():
        _ = model(x)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n):
            _ = model(x)
    return (time.perf_counter() - t0) / n * 1000

t_rnn  = bench(rnn_bench,  x_bench)
t_lstm = bench(lstm_bench, x_bench)
t_attn = bench(attn_bench, x_bench)

print(f"Single forward pass timing (B={BATCH_SIZE}, T={SEQ_LEN}, D={D_MODEL}):")
print(f"  RNN            : {t_rnn:.2f} ms")
print(f"  LSTM           : {t_lstm:.2f} ms")
print(f"  Self-Attention : {t_attn:.2f} ms")
print(f"\n  Attention / RNN  speedup : {t_rnn/t_attn:.2f}x (attention is parallelised)")
print(f"  LSTM   / RNN   ratio     : {t_rnn/t_lstm:.2f}x")
print(f"\nNote: on GPU, attention and parallel models benefit much more from hardware.")
print(f"Mamba/S4 would be O(T) with parallel scan — better than both for long T.")
